# 05 · Aktualizacje wielokrokowe i ślady

**Punkt wyjścia.** Znamy cel $R_{t+1}+\gamma V(S_{t+1})$ z TD(0), a z Monte Carlo pełny zdyskontowany zwrot do końca epizodu. Ile zaobserwowanych nagród warto uwzględnić przed użyciem oszacowania dalszej wartości?

1. Policz trzy cele dla jednego odcinka na lodzie.
2. Zaimplementuj cel $n$-krokowy i rozpoznaj go w ważonej sumie ze współczynnikiem $\lambda$.
3. Prześledź aktualizację stanu odwiedzanego po raz drugi. Porównaj własny TD($\lambda$) z TD(0) względem wartości obliczonych z modelu.

Uruchamiaj komórki od góry. Plik `srodowiska.py` powinien znajdować się obok. Wystarczą Python 3 i środowisko Jupyter z jądrem Pythona.

## 1. Jedna trajektoria i trzy cele

Znana plansza 4×4 z poślizgiem. Przyjmij $\gamma=0{,}95$, $V(14)=0{,}60$ i $V(10)=0{,}40$. Poniżej nagroda należy do **przejścia**, a stan 15 jest celem kończącym epizod w chwili $T=3$.

| Czas $t$ | Stan $S_t$ | Faktyczna komenda | Następny stan $S_{t+1}$ | Nagroda $R_{t+1}$ |
|---:|---:|---|---:|---:|
| 0 | 14 | prawo | 10 | 0 |
| 1 | 10 | dół | 14 | 0 |
| 2 | 14 | prawo | 15 | +1 |

Dla $m=\min(n,T-t)$ cel wynosi
$$
G_t^{(n)}=\sum_{j=0}^{m-1}\gamma^j R_{t+j+1}
+\begin{cases}
\gamma^nV(S_{t+n}) & \text{gdy } t+n<T,\\
0 & \text{gdy } t+n\ge T.
\end{cases}
$$

**Ćwiczenie rachunkowe 1.** Zapisz $G_0^{(1)}$, $G_0^{(2)}$ oraz $G_0^{(3)}$. Przy każdym celu zaznacz osobno część z obserwacji i część z aktualnej tablicy $V$. Co otrzymasz dla $n=8$? Dlaczego w kroku dwukrokowym wraca stan 14?

In [ ]:
from math import isclose, sqrt
from random import Random
from statistics import median
from srodowiska import FrozenLake

GAMMA = 0.95
STANY = (14, 10, 14, 15)
NAGRODY = (0.0, 0.0, 1.0)
V_POCZATKOWE = [0.0] * 16
V_POCZATKOWE[14] = 0.60
V_POCZATKOWE[10] = 0.40

### Python · cel $n$-krokowy

Argument `stany` zawiera $T+1$ pozycji. Element `nagrody[t]` oznacza $R_{t+1}$, czyli nagrodę za przejście ze stanu `stany[t]`. `t` ma zakres $0,\ldots,T-1$.

- Ustal koniec części obserwowanej, `koniec = min(t + n, T)`.
- Zsumuj nagrody od indeksu `t` do `koniec - 1`, z wykładnikami $0,1,\ldots$.
- Jeżeli `t + n < T`, dopisz dyskontowaną wartość stanu w chwili `t + n`.

In [ ]:
def cel_n_krokowy(stany, nagrody, t, n, gamma, wartosci):
    # TODO: zsumuj m nagród i w odpowiednim przypadku dodaj wartość dalszą.
    raise NotImplementedError("Uzupełnij cel n-krokowy")

In [ ]:
def blisko(wynik, oczekiwany):
    assert isclose(wynik, oczekiwany, abs_tol=1e-12), (wynik, oczekiwany)


for n, oc in [(1, 0.38), (2, 0.5415), (3, 0.9025), (8, 0.9025)]:
    cel = cel_n_krokowy(STANY, NAGRODY, 0, n, GAMMA, V_POCZATKOWE)
    blisko(cel, oc)
    print(f"t=0, n={n}: cel = {cel:.4f}")

# Test zakończenia: sztucznie wpisujemy dużą wartość celu do tablicy.
# Wejście do terminala ma nadal zwrot 1, a nie 1 + γ × 999.
wartosci_z_pulapka = V_POCZATKOWE.copy()
wartosci_z_pulapka[15] = 999.0
blisko(cel_n_krokowy(STANY, NAGRODY, 2, 1, GAMMA, wartosci_z_pulapka), 1.0)

## 2. Zwrot $\lambda$ w perspektywie do przodu

W tym ćwiczeniu tablica $V$ pozostaje zamrożona podczas liczenia wszystkich celów. Dla $m=T-t$ i $0\le\lambda\le1$ użyj rozkładu wag
$$
G_t^\lambda=(1-\lambda)\sum_{n=1}^{m-1}\lambda^{n-1}G_t^{(n)}
+\lambda^{m-1}G_t^{(m)}.
$$
Waga pełnego zwrotu, $\lambda^{m-1}$, skupia pozostałą masę i domyka sumę wag do 1. Gdy pozostał jeden krok, pełny zwrot ma wagę 1. Dla $\lambda=0$ dostajemy cel TD(0), a dla $\lambda=1$ pełny zwrot do końca epizodu. Tak liczony pełny zwrot dotyczy wskazanego wystąpienia stanu. O tym, czy MC aktualizuje tylko pierwszą, czy każdą wizytę, decyduje osobna reguła.

**Ćwiczenie rachunkowe 2.** Jakie są trzy wagi dla $t=0$ przy $\lambda=0{,}5$? Skorzystaj z wyników ćwiczenia 1. Następnie policz cel dla $t=2$ wobec tej samej wartości $\lambda$.

In [ ]:
def cel_lambda(stany, nagrody, t, gamma, lam, wartosci):
    m = len(nagrody) - t
    cele_krotsze = (
        (1 - lam) * lam ** (n - 1)
        * cel_n_krokowy(stany, nagrody, t, n, gamma, wartosci)
        for n in range(1, m)
    )
    pelny = cel_n_krokowy(stany, nagrody, t, m, gamma, wartosci)
    return sum(cele_krotsze) + lam ** (m - 1) * pelny


for lam, oc in [(0.0, 0.38), (0.5, 0.551), (1.0, 0.9025)]:
    cel = cel_lambda(STANY, NAGRODY, 0, GAMMA, lam, V_POCZATKOWE)
    blisko(cel, oc)
    print(f"λ={lam:.1f}: cel dla t=0 wynosi {cel:.4f}")
blisko(cel_lambda(STANY, NAGRODY, 2, GAMMA, 0.5, V_POCZATKOWE), 1.0)

## 3. TD($\lambda$) ze śladami akumulującymi

W wersji online po każdym ruchu obliczamy błąd TD i przydzielamy go stanom, które właśnie odwiedziliśmy lub odwiedzaliśmy niedawno. Ślad $e_t(s)$ określa siłę tego przydziału.

$$
\begin{aligned}
\delta_t &= R_{t+1}+\gamma V(S_{t+1})-V(S_t),\\
e_t(s) &= \gamma\lambda e_{t-1}(s)+\mathbf{1}[s=S_t],\\
V(s) &\gets V(s)+\alpha\delta_t e_t(s).
\end{aligned}
$$

W przejściu kończącym epizod w $\delta_t$ podstawiamy dalszą wartość równą 0. $\alpha$ to krok uczenia, a $\mathbf{1}[s=S_t]$ dodaje 1 wyłącznie do śladu aktualnego stanu. Zerujemy wszystkie ślady na początku każdego epizodu. W kolejnej wizycie w tym samym stanie ślad akumulujący dodaje 1 do swojej pozostałości. Ślad zastępujący w tej sytuacji ustawiłby go na 1.

**Ćwiczenie rachunkowe 3.** Przy $\alpha=0{,}10$, $\gamma=0{,}95$ i $\lambda=0{,}5$ wyznacz ślad stanu 14 po pierwszym, drugim i trzecim ruchu. Oblicz pierwszy błąd $\delta_0$ oraz nową wartość 14. W drugim ruchu oblicz błąd ze zaktualizowanego $V(14)$.

### Python · jeden krok śladów

Policz $\delta$ z wartości sprzed tej aktualizacji. Pomnóż wszystkie ślady przez $\gamma\lambda$, dodaj wizytę aktualnego stanu, a potem przesuń wartości o $\alpha\delta e$. Funkcja modyfikuje przekazane listy i zwraca błąd $\delta$.

In [ ]:
def krok_td_lambda(wartosci, slady, s, r, sp, koniec, alfa, gamma, lam):
    # TODO: wyznacz δ, zaktualizuj ślady, przesuń wartości i zwróć δ.
    raise NotImplementedError("Uzupełnij TD ze śladami")

In [ ]:
def przejdz_reczny_epizod(lam):
    v = V_POCZATKOWE.copy()
    slady = [0.0] * len(v)
    tabela = []
    for t, r in enumerate(NAGRODY):
        s, sp = STANY[t:t + 2]
        delta = krok_td_lambda(
            v, slady, s, r, sp, t + 1 == len(NAGRODY), 0.1, GAMMA, lam
        )
        tabela.append((delta, slady[14], slady[10], v[14], v[10]))
    return tabela


tabela = przejdz_reczny_epizod(0.5)
print("t      δ        e(14)      e(10)      V(14)      V(10)")
for t, wiersz in enumerate(tabela):
    print(f"{t}   " + "  ".join(f"{x:9.6f}" for x in wiersz))
blisko(tabela[0][0], -0.22)
blisko(tabela[0][3], 0.578)
blisko(tabela[1][0], 0.1491)
blisko(tabela[1][1], 0.475)
blisko(tabela[2][1], 1.225625)

# Sprawdź granicę TD(0), niezależnie od wzoru na ślady.
referencja = V_POCZATKOWE.copy()
for t, r in enumerate(NAGRODY):
    s, sp = STANY[t:t + 2]
    dalsza = referencja[sp] if t + 1 < len(NAGRODY) else 0.0
    referencja[s] += 0.1 * (r + GAMMA * dalsza - referencja[s])
ostatni_wiersz = przejdz_reczny_epizod(0.0)[-1]
blisko(ostatni_wiersz[3], referencja[14])
blisko(ostatni_wiersz[4], referencja[10])

### Jak połączyć obie perspektywy?

Dla pojedynczego wystąpienia stanu, przy **tej samej zamrożonej tablicy** $V$, zachodzi
$$
G_t^\lambda - V(S_t)=\sum_{j=t}^{T-1}(\gamma\lambda)^{j-t}\delta_j.
$$
Ślad zapewnia ten sam rozkład wag, gdy wszystkie błędy liczymy z zamrożonych wartości i sumujemy aktualizacje po epizodzie. W uruchomionej powyżej wersji online $V$ zmienia się po każdym przejściu. Na przykład drugi błąd TD korzysta już z $V(14)=0{,}578$. Porównanie dwóch perspektyw wymaga więc wskazania, z której tablicy korzysta każda aktualizacja.

In [ ]:
def sprawdz_tozsamosc_na_zamrozonej_tablicy(lam):
    delty = [
        r + (GAMMA * V_POCZATKOWE[STANY[t + 1]] if t + 1 < len(NAGRODY) else 0.0)
        - V_POCZATKOWE[STANY[t]]
        for t, r in enumerate(NAGRODY)
    ]
    for t, s in enumerate(STANY[:-1]):
        lewa = cel_lambda(STANY, NAGRODY, t, GAMMA, lam, V_POCZATKOWE) - V_POCZATKOWE[s]
        prawa = sum((GAMMA * lam) ** (j - t) * delty[j] for j in range(t, len(delty)))
        blisko(lewa, prawa)


for lam in (0.0, 0.5, 0.9, 1.0):
    sprawdz_tozsamosc_na_zamrozonej_tablicy(lam)
print("Sprawdzono tożsamość na zamrożonej tablicy.")

## 4. Eksperyment na tej samej śliskiej planszy

Oceniamy **jedną stałą politykę**, wybierającą każdą komendę z szansą $1/4$. To inna polityka niż decyzje z ręcznego przykładu. Do punktu odniesienia użyjemy iteracyjnej oceny z pełnym modelem przejść. Uczące się metody dostaną te same zarejestrowane epizody i ich nagrody. Nie potrzebują listy niewylosowanych skutków.

Porównaj TD(0), czterokrokowy TD i TD($\lambda$) dla $\lambda=0{,}5$ oraz $0{,}9$. Wszystkie konfiguracje zaczynają od zera i mają $\alpha=0{,}1$, $\gamma=0{,}95$. Wyświetlamy medianę po pięciu ziarnach. Błąd to pierwiastek ze średniej kwadratów różnic wartości pól nieterminalnych względem modelu.

W implementacji czterokrokowej mamy już zarejestrowany epizod i odtwarzamy aktualizacje w kolejności odwiedzin, używając tylko czterech dostępnych nagród lub krótszej końcówki. Przy zbieraniu doświadczenia na żywo pierwsza taka aktualizacja może nastąpić po czterech przejściach albo po wcześniejszym końcu epizodu.

In [ ]:
def ocen_z_modelu(env, gamma):
    v = [0.0] * env.n_states
    for _ in range(10_000):
        nowa = [
            sum(
                p * (r + (0.0 if done else gamma * v[sp]))
                for a in env.actions
                for p, sp, r, done in env.outcomes(s, a)
            ) / len(env.actions)
            if not env.terminal(s) else 0.0
            for s in range(env.n_states)
        ]
        if max(abs(a - b) for a, b in zip(v, nowa)) < 1e-13:
            return nowa
        v = nowa
    raise RuntimeError("Ocena polityki nie osiągnęła tolerancji")


def zbierz_epizod(env, rng_akcji, rng_lodu):
    stany = [env.start]
    nagrody = []
    for _ in range(4_000):
        a = rng_akcji.choice(env.actions)
        sp, r, koniec = env.sample(stany[-1], a, rng_lodu)
        stany.append(sp)
        nagrody.append(r)
        if koniec:
            return stany, nagrody
    raise RuntimeError("Przekroczono limit rejestracji pełnego epizodu")


def ucz_n_krokowo(v, stany, nagrody, n, gamma, alfa):
    for t, s in enumerate(stany[:-1]):
        cel = cel_n_krokowy(stany, nagrody, t, n, gamma, v)
        v[s] += alfa * (cel - v[s])


def ucz_sladowo(v, stany, nagrody, lam, gamma, alfa):
    slady = [0.0] * len(v)
    for t, r in enumerate(nagrody):
        s, sp = stany[t:t + 2]
        krok_td_lambda(v, slady, s, r, sp, t + 1 == len(nagrody), alfa, gamma, lam)


def rmse(v, oczekiwane, pola):
    return sqrt(sum((v[s] - oczekiwane[s]) ** 2 for s in pola) / len(pola))

In [ ]:
def porownaj(ziarna, etapy):
    env = FrozenLake(slippery=True)
    ref = ocen_z_modelu(env, GAMMA)
    pola = [s for s in range(env.n_states) if not env.terminal(s)]
    etykiety = ("TD(0)", "4 kroki", "TD(λ=0,5)", "TD(λ=0,9)")
    bledy = {(ep, nazwa): [] for ep in etapy for nazwa in etykiety}
    dojscia = {ep: [] for ep in etapy}
    for seed in ziarna:
        rng_akcji = Random(seed)
        rng_lodu = Random(seed + 10_000)
        tablice = [[0.0] * env.n_states for _ in etykiety]
        do_celu = 0
        for ep in range(1, max(etapy) + 1):
            stany, nagrody = zbierz_epizod(env, rng_akcji, rng_lodu)
            do_celu += nagrody[-1] == 1.0
            ucz_n_krokowo(tablice[0], stany, nagrody, 1, GAMMA, 0.10)
            ucz_n_krokowo(tablice[1], stany, nagrody, 4, GAMMA, 0.10)
            ucz_sladowo(tablice[2], stany, nagrody, 0.5, GAMMA, 0.10)
            ucz_sladowo(tablice[3], stany, nagrody, 0.9, GAMMA, 0.10)
            if ep in etapy:
                dojscia[ep].append(do_celu)
                for nazwa, v in zip(etykiety, tablice):
                    bledy[ep, nazwa].append(rmse(v, ref, pola))
    print(f"Wartość referencyjna startu: {ref[env.start]:.5f}")
    print("Epizody  Mediana dojść" + "".join(f"{nazwa:>14}" for nazwa in etykiety))
    for ep in etapy:
        print(f"{ep:7d}{median(dojscia[ep]):15.0f}" + "".join(f"{median(bledy[ep, nazwa]):14.5f}" for nazwa in etykiety))
    return bledy


bledy = porownaj(ziarna=(2, 7, 19, 37, 71), etapy=(25, 100, 500))

**Do oddania.** Wybierz dwa etapy i porównaj błąd dwóch metod. Czy ranking zmienia się z liczbą epizodów? Sprawdź w tabeli, kiedy w większości przebiegów agent po raz pierwszy widzi nagrodę dodatnią. Zajrzyj do list `bledy[(100, "TD(0)")]` i `bledy[(100, "TD(λ=0,9)")]`. Wskaż dowolne dwa ziarna, na których porównanie wygląda podobnie albo inaczej. Jak dochodzące kolejne wizyty i stały krok uczenia mogą wyjaśniać wahania? Przy zmianie $n$ albo $\lambda$ pozostałe ustawienia zostaw na poziomie bazowym.

## 5. Przejście do wielokrokowego SARSA

W predykcji zapisujemy $V(s)$. W sterowaniu zapisujemy $Q(s,a)$ i przechowujemy również wykonaną akcję na każdym kroku. W wersji $n$-krokowego SARSA do celu dochodzi $\gamma^n Q(S_{t+n},A_{t+n})$, o ile $t+n<T$. $A_{t+n}$ jest akcją **rzeczywiście wybraną przez aktualną politykę zachowania**. Wcześniejsze zakończenie usuwa składnik dalszej wartości.

**Ćwiczenie rozszerzające.** Na naszej trajektorii komendy są kolejno „w prawo”, „w dół”, „w prawo”. Przyjmij $Q(10,\text{dół})=0{,}20$ i $Q(14,\text{prawo})=0{,}55$. Policz cel jedno-, dwu- i trzykrokowy dla pary $(14,\text{prawo})$ z pierwszej wizyty. Następnie uruchom komórkę. Wariant śladowy SARSA($\lambda$) używa tej samej aktualizacji śladów, z indeksowaniem parą $(s,a)$ i błędem SARSA z rozdziału 04.

In [ ]:
# Te same pozycje czasowe. Ostatnia para jest terminalna i nie potrzebuje akcji.
pary = ((14, 2), (10, 1), (14, 2), (15, None))
q = {(14, 2): 0.55, (10, 1): 0.20}
for n, oc in [(1, 0.19), (2, 0.496375), (3, 0.9025)]:
    cel = cel_n_krokowy(pary, NAGRODY, 0, n, GAMMA, q)
    blisko(cel, oc)
    print(f"SARSA, n={n}: cel = {cel:.6f}")